# 01 - Exploratory analysis: TalkingData click fraud

EDA only. Anything that survives into production lives in `adshield.features.engineering`.

Label: `is_attributed == 1` is a real conversion; `0` never converted and is used as a fraud-risk **proxy**.

Set `ADSHIELD_DATA` / `ADSHIELD_NROWS` to override the file or sample size. The full file is ~184M rows; the first N rows are time-ordered, so a head sample covers only the start of the window.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from adshield.data.loader import load_clicks, dataset_health
from adshield.features.engineering import velocity_features

DATA = Path(os.environ.get("ADSHIELD_DATA", "../data/raw/train.csv"))
NROWS = int(os.environ["ADSHIELD_NROWS"]) if os.environ.get("ADSHIELD_NROWS") else 5_000_000
df = load_clicks(DATA, nrows=NROWS)
print(dataset_health(df))

## 1. Class imbalance

In [ ]:
counts = df["is_attributed"].value_counts().sort_index()
pct_real = 100 * counts.get(1, 0) / len(df)
print(counts.to_string())
print(f"real: {pct_real:.4f}%  |  non-converting ('fraud' proxy): {100 - pct_real:.4f}%")
print(f"imbalance ratio (majority:minority) = {counts.max() / max(counts.min(), 1):,.0f}:1")
counts.plot.bar(logy=True, title="Class counts (log scale)"); plt.show()

## 2. Click bursts per IP

In [ ]:
per_ip = df.groupby("ip").size().rename("clicks")
print(per_ip.describe(percentiles=[.5, .9, .99, .999]).to_string())
per_ip.clip(upper=per_ip.quantile(.999)).plot.hist(bins=60, logy=True, title="Clicks per IP (clipped at p99.9)"); plt.show()

In [ ]:
vel = velocity_features(df, {"clicks_10min": 600})
df_v = df.assign(burst_10min=vel["clicks_10min"].to_numpy())
print(df_v["burst_10min"].describe(percentiles=[.5, .9, .99, .999]).to_string())

bins = [0, 1, 2, 5, 10, 25, 50, np.inf]
bucket = pd.cut(df_v["burst_10min"], bins=bins)
by_burst = df_v.groupby(bucket, observed=True)["is_attributed"].agg(clicks="size", conv_rate="mean")
print(by_burst)
by_burst["conv_rate"].plot.bar(title="Conversion rate by clicks from same IP in trailing 10 min"); plt.show()

In [ ]:
# Click timeline of the busiest IP, per minute
top_ip = per_ip.idxmax()
s = df.loc[df["ip"] == top_ip, "click_time"]
s.dt.floor("1min").value_counts().sort_index().plot(title=f"Clicks per minute, IP {top_ip}"); plt.show()

## 3. Time between clicks per IP

Humans have irregular, long gaps. Scripts show many identical or near-zero gaps and low variability.

In [ ]:
d = df.sort_values(["ip", "click_time"], kind="stable")
gap = d.groupby("ip")["click_time"].diff().dt.total_seconds()
d = d.assign(gap_s=gap)
g = d["gap_s"].dropna()
print(g.describe(percentiles=[.1, .5, .9, .99]).to_string())
print(f"share of gaps == 0s: {(g == 0).mean():.2%}  |  <= 1s: {(g <= 1).mean():.2%}")
np.log10(g.clip(lower=1)).plot.hist(bins=60, logy=True, title="log10(seconds between consecutive clicks, same IP)"); plt.show()

In [ ]:
stats = d.groupby("ip")["gap_s"].agg(["count", "mean", "std"])
stats = stats[stats["count"] >= 5]
stats["cv"] = stats["std"] / stats["mean"].replace(0, np.nan)   # low CV = metronomic
conv = df.groupby("ip")["is_attributed"].mean().rename("conv_rate")
stats = stats.join(conv)
stats["cv_bucket"] = pd.qcut(stats["cv"], 5, duplicates="drop")
print(stats.groupby("cv_bucket", observed=True)["conv_rate"].agg(["size", "mean"]))

## 4. Categorical combinations vs label

In [ ]:
base = df["is_attributed"].mean()
print(f"baseline conversion rate: {base:.5f}")
MIN_SUPPORT = 500

def combo_table(cols):
    t = df.groupby(cols, observed=True)["is_attributed"].agg(clicks="size", conv_rate="mean").reset_index()
    t = t[t["clicks"] >= MIN_SUPPORT].copy()
    t["lift_vs_base"] = t["conv_rate"] / base
    # non-converting clicks beyond what the baseline conversion rate would predict
    t["excess_nonconv"] = t["clicks"] * (base - t["conv_rate"])
    return t

for cols in (["app"], ["channel"], ["os"], ["device"]):
    print(f"\n== {cols}: lowest conversion, min support {MIN_SUPPORT} ==")
    print(combo_table(cols).sort_values("conv_rate").head(10).to_string(index=False))

In [ ]:
full = combo_table(["app", "device", "os", "channel"])
print("Lowest conversion combos:"); print(full.sort_values("conv_rate").head(20).to_string(index=False))
print("\nLargest excess non-converting volume:"); print(full.sort_values("excess_nonconv", ascending=False).head(20).to_string(index=False))

## Findings

_Fill in after running on the real file (class imbalance severity, burst thresholds, gap regularity, riskiest combinations). Not written from synthetic data._